## Simple Tool Calls
### Tools
**In LangChain Python, tools are created using the @tool decorator**
* Tools are nothing but functions/methods with proper defined input and output and description.
* These decriptions / docstrings added to functions helps LLM to understand what a Function/Tool does.
* Adding Pydantic schema as tool description allow LLM to understand about its strict schema.
* This function converted to Tool with @tool gives more context to LLM about the existing tools to it.

In [2]:
import os
import json
from dotenv import load_dotenv
from typing import Literal
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv()

True

In [4]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

model = ChatOpenAI(model="gpt-5-nano")

#### Define Pydantic schema for a Tool
* For complex inputs, define the schema with a Pydantic model instead of inline type hints alone to give richer context.
* SeatBookingInput decribes the structure of tool `book_seats` defined below.
* This tool takes parameter movie_title, seat_count & preferred_row. 

In [5]:
class SeatBookingInput(BaseModel):
    """Input for booking cinema seats."""
    movie_title: str = Field(description="Exact movie title")
    seat_count: int = Field(description="Number of seats to book", ge=1, le=10)
    preferred_row: Literal["front", "middle", "back"] = Field(default="middle", description="Preferred seating row")



### Defining a Tool
<img src="../../assets/tools_definition.png" width="800" height="300">

* Description in @tool decorator parameter overrides the docstring description.
* Langchain provides a way to define docstring and tool description separately, but is not provided in @tool, it will use docstring as desciption.
* We have created 2 tools `check_showtimes` and `book_seats`.
    * **check_showtimes**: This tool is defined Overriding the Name and Description in @tool decorator.
    * **book_seats** Advanced Input Schemas using `args_schema` that takes Pydantic model for schema interpretation.
* Here we also define `tools_name_func_map` that will help to make actual call to tools from our code.

In [11]:
@tool()
def check_showtimes(movie_title: str) -> str:
  """Check available showtimes for a movie at the cinema.
  Args:
      movie_title: The exact title of the movie to check
  """
  fake_showtimes = {
      "interstellar": "7:00 PM and 10:15 PM",
      "dune part two": "9:30 PM only",
      "oppenheimer": "Sold out for tonight",
  }
  return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

@tool(args_schema=SeatBookingInput, description = 'Book tickets for a customer, use whenever customer wants to book/reserve a seat.')
def book_seats(movie_title: str, seat_count: int, preferred_row: str = "middle") -> str:
    """Book seats for a movie."""
    return f"Booked {seat_count} seat(s) in the {preferred_row} row for {movie_title}."

print("check_showtimes: \n", json.dumps(check_showtimes.args, indent=2))
print("reserve: Advanced Input Schema -------- \n", json.dumps(book_seats.args, indent=2))

tools_name_func_map = {'book_seats': book_seats, 'check_showtimes': check_showtimes}

check_showtimes: 
 {
  "movie_title": {
    "title": "Movie Title",
    "type": "string"
  }
}
reserve: Advanced Input Schema -------- 
 {
  "movie_title": {
    "description": "Exact movie title",
    "title": "Movie Title",
    "type": "string"
  },
  "seat_count": {
    "description": "Number of seats to book",
    "maximum": 10,
    "minimum": 1,
    "title": "Seat Count",
    "type": "integer"
  },
  "preferred_row": {
    "default": "middle",
    "description": "Preferred seating row",
    "enum": [
      "front",
      "middle",
      "back"
    ],
    "title": "Preferred Row",
    "type": "string"
  }
}


#### STEP 1: LLM Geneates Tool Call
* The LLM's role: Analyze the request and decide which tool to call
* Important: The LLM does NOT execute anything - it just generates a plan!

In [12]:
model_with_tools = model.bind_tools([check_showtimes, book_seats])
query = "Is Interstellar showing tonight at 7pm at the Downtown cinema ?"
response = model_with_tools.invoke(query)

#### LLM response
* LLM responded to make a tool call with below information.
    1. Name
    2. Arguments
    3. Id 

In [13]:
print(response.content)
tool_call = response.tool_calls[0]
print("LLM decided to call:", tool_call["name"])
print("With arguments:", tool_call["args"])
print("Tool call ID:", tool_call["id"])


LLM decided to call: check_showtimes
With arguments: {'movie_title': 'Interstellar'}
Tool call ID: call_UfA2S9lWX5Z2lhwc44nltWkJ


#### STEP 2: Our Code Executes The Tool
* Our code actually execute the function and get real results
* This is where the real work happens - API calls, database queries, etc.
* Here we make call to tool `check_showtimes` using .invoke() method.
* Tool responded with return statement of check_showtimes. 

In [14]:
tool_result = tools_name_func_map[tool_call["name"]].invoke(tool_call["args"])
print("Tool executed successfully!")
print("Tool result:", tool_result)

Tool executed successfully!
Tool result: 7:00 PM and 10:15 PM


### STEP 3: Send Results Back To LLM.
* Create a messages list that contains our LLM conversation.
* Include 
    * HumanMessage with our query as content.
    * AIMessage with LLM response content and suggested tool to make call on.
    * ToolMessage with Tool Results and the tool call id.
* Give this information back to LLM to get the refined Natural Language response.

In [15]:
messages = [
    HumanMessage(content=query),
    AIMessage(content=str(response.content), tool_calls=response.tool_calls),
    ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
]

#### Refined Final response
* Final reponse by LLM with information about available show time for a movie.
* **Also it responded with option to further reserve tickets, this is because LLM knows it has the capability (Tool) to reserve ticket**

In [17]:
final_response = model.invoke(messages)
print(final_response.content)

Yes. Interstellar is showing tonight at the Downtown cinema at 7:00 PM. There’s also a second showing at 10:15 PM if you’d prefer that.

Would you like me to reserve tickets? If so, how many seats and which showtime?


## You Don't Always Need to Write a Tool Yourself: 
#### Langchain [Prebuilt Tools](https://docs.langchain.com/oss/python/integrations/tools)
* Before writing a custom tool for something common — like web search
* Check if a ready-made package already exists. 
* E.g. Tavily is a widely-used search tool built specifically for AI agents, available as a LangChain-compatible tool with almost no setup.